# Robustness and Validation Checks (Section 4.8)

Run the MASTER SETUP CELL first, then any check. F5 and F6 need F4 to be run first. `DATA.xlsx` must be in the same folder. Random seed 42.

## Master setup cell

In [ ]:
# ── MASTER SETUP CELL — Run this first after any session reset ──
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import os

# ── Load data ──
df_flights = pd.read_excel("DATA.xlsx", sheet_name="Flight Records")
df_monthly = pd.read_excel("DATA.xlsx", sheet_name="Monthly Summary")
df_annual  = pd.read_excel("DATA.xlsx", sheet_name="Annual Summary")

for col in ['TOTAL','AUW','YEAR','MONTH']:
    if col in df_flights.columns:
        df_flights[col] = pd.to_numeric(df_flights[col], errors='coerce')
df_flights = df_flights[df_flights['TOTAL'] > 0].reset_index(drop=True)

df_monthly['REVENUE'] = pd.to_numeric(df_monthly['REVENUE'], errors='coerce')
df_monthly['YEAR']    = pd.to_numeric(df_monthly['YEAR'],    errors='coerce')
df_monthly['MONTH']   = pd.to_numeric(df_monthly['MONTH'],   errors='coerce')
df_monthly['FLIGHTS'] = pd.to_numeric(df_monthly['FLIGHTS'], errors='coerce')

annual_revenue = df_flights.groupby('YEAR')['TOTAL'].sum()

# ── Inflation data ──
inflation_data = {
    2008:0.1158, 2009:0.1254, 2010:0.1374, 2011:0.1083,
    2012:0.1222, 2013:0.0850, 2014:0.0805, 2015:0.0901,
    2016:0.1570, 2017:0.1650, 2018:0.1210, 2019:0.1140,
    2020:0.1325, 2021:0.1695, 2022:0.1885, 2023:0.2466,
    2024:0.3324, 2025:0.2333
}
inflation_df = pd.DataFrame(list(inflation_data.items()),
                             columns=['YEAR','INFLATION_RATE'])

def cumulative_factor_lagged(year):
    factor = 1.0
    for yr in range(2008, year):
        if yr in inflation_data:
            factor *= (1 + inflation_data[yr])
    return factor

mean_inflation        = inflation_df['INFLATION_RATE'].mean()
std_inflation         = inflation_df['INFLATION_RATE'].std()
mean_future_inflation = mean_inflation
std_future_inflation  = std_inflation

# ── Simulation base parameters ──
base_revenue      = annual_revenue[2025] * (12/9)
base_movements    = df_monthly[df_monthly['YEAR']==2025]['FLIGHTS'].sum() * (12/9)
avg_charge_per_flight = base_revenue / base_movements
A_base            = cumulative_factor_lagged(2025)

actual_2021 = df_monthly[df_monthly['YEAR']==2021]['ACTUAL_FLIGHTS'].sum()
actual_2024 = df_monthly[df_monthly['YEAR']==2024]['ACTUAL_FLIGHTS'].sum()
cagr_movements = (actual_2024 / actual_2021) ** (1/3) - 1

proj_years = [2026, 2027, 2028, 2029, 2030]
g_values   = [0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]
SUSTAINABILITY_THRESHOLD = 0.80
N_ITERATIONS = 10000

# ── Annual actual revenue for retrospective ──
annual_actual = df_monthly.groupby('YEAR')['REVENUE'].sum().reset_index()
annual_actual.columns = ['YEAR','ACTUAL_REVENUE']
annual_actual.loc[annual_actual['YEAR']==2025,'ACTUAL_REVENUE'] *= (12/9)
anchor_2021 = annual_actual.loc[annual_actual['YEAR']==2021,'ACTUAL_REVENUE'].values[0]
annual_actual['INFLATION_FACTOR'] = annual_actual['YEAR'].apply(
    lambda yr: cumulative_factor_lagged(int(yr)))
annual_actual['ADJUSTED_REVENUE'] = anchor_2021 * annual_actual['INFLATION_FACTOR']
annual_actual['REVENUE_GAP']      = annual_actual['ADJUSTED_REVENUE'] - annual_actual['ACTUAL_REVENUE']
annual_actual['GAP_PERCENT']      = (annual_actual['REVENUE_GAP'] /
                                      annual_actual['ADJUSTED_REVENUE']) * 100

# ── Inflation-required charge by 2030 ──
future_factor      = (1 + mean_future_inflation) ** 5
full_factor_2030   = cumulative_factor_lagged(2026) * future_factor
required_charge_2030 = avg_charge_per_flight * full_factor_2030

print("✅ Setup complete")
print(f"Base revenue:          ₦{base_revenue/1e6:.2f}M")
print(f"Avg charge/flight:     ₦{avg_charge_per_flight:,.0f}")
print(f"Movement CAGR:         {cagr_movements*100:.2f}%")
print(f"A_base (2025):         {A_base:.4f}")
print(f"Mean inflation:        {mean_inflation*100:.2f}%")
print(f"Std inflation:         {std_inflation*100:.2f}%")
print(f"Required charge 2030:  ₦{required_charge_2030:,.0f}")


## F1: Monte Carlo convergence check (Table 4.17, Figure 4.9)

In [ ]:
# -- CELL: Monte Carlo Convergence Check --
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df_flights = pd.read_excel("DATA.xlsx", sheet_name="Flight Records")
df_monthly = pd.read_excel("DATA.xlsx", sheet_name="Monthly Summary")
df_flights = df_flights[pd.to_numeric(df_flights['TOTAL'], errors='coerce') > 0]

inflation_data = {
    2008:0.1158, 2009:0.1254, 2010:0.1374, 2011:0.1083,
    2012:0.1222, 2013:0.0850, 2014:0.0805, 2015:0.0901,
    2016:0.1570, 2017:0.1650, 2018:0.1210, 2019:0.1140,
    2020:0.1325, 2021:0.1695, 2022:0.1885, 2023:0.2466,
    2024:0.3324, 2025:0.2333
}
rates_all = np.array(list(inflation_data.values()))
mu = rates_all.mean()
sd = rates_all.std(ddof=0)          # population SD, same as Cell 2 (6.29%)

def cumulative_factor_lagged(year):
    return np.prod([1 + inflation_data[y] for y in range(2008, year)])

A_base = cumulative_factor_lagged(2025)
anchor_2021 = df_monthly[df_monthly['YEAR'] == 2021]['REVENUE'].sum()
base_movements = df_monthly[df_monthly['YEAR'] == 2025]['FLIGHTS'].sum() * 12/9
base_revenue = df_flights[df_flights['YEAR'] == 2025]['TOTAL'].sum() * 12/9
avg_charge = base_revenue / base_movements
a21 = df_monthly[df_monthly['YEAR'] == 2021]['ACTUAL_FLIGHTS'].sum()
a24 = df_monthly[df_monthly['YEAR'] == 2024]['ACTUAL_FLIGHTS'].sum()
cagr = (a24 / a21) ** (1/3) - 1

# ---- Model outputs (vectorised versions of Cells 7, 11 and 12) ----
def retro_total(N, rng):
    total = np.zeros(N)
    for year in range(2021, 2026):
        r = np.array([inflation_data[y] for y in range(2008, year)])
        s = np.clip(rng.normal(r, 0.05, (N, len(r))), 0.01, 0.80)
        total += anchor_2021 * (np.prod(1 + s, axis=1) - 1)
    return total

def gap_pct(N, rng, g, n_years):
    fi = np.clip(rng.normal(mu, sd, (N, n_years)), 0.01, 0.80)
    return 1 - (1 + g) ** n_years / (A_base * np.prod(1 + fi, axis=1))

def s1_gap_naira_2030(N, rng):
    fi = np.clip(rng.normal(mu, sd, (N, 5)), 0.01, 0.80)
    m = base_movements * (1 + cagr) ** 5
    return m * avg_charge * (A_base * np.prod(1 + fi, axis=1) - 1)

metrics = {
    'Retro total gap 2021-2025 (N M)': lambda N, rng: retro_total(N, rng).mean() / 1e6,
    'S1 2030 mean gap (N M)':          lambda N, rng: s1_gap_naira_2030(N, rng).mean() / 1e6,
    'S1 2030 gap %':                   lambda N, rng: gap_pct(N, rng, 0.00, 5).mean() * 100,
    'S3 2030 P(gap>80%) %':            lambda N, rng: (gap_pct(N, rng, 0.30, 5) > 0.8).mean() * 100,
    'S4 2028 P(gap>80%) %':            lambda N, rng: (gap_pct(N, rng, 0.50, 3) > 0.8).mean() * 100,
}

# ---- 30 independent runs at each simulation size ----
N_list = [500, 1000, 2000, 5000, 10000, 20000, 50000]
R = 30
rows = []
for name, f in metrics.items():
    for N in N_list:
        vals = np.array([f(N, np.random.default_rng(1000 + i)) for i in range(R)])
        rows.append([name, N, vals.mean(), vals.std(ddof=1)])

conv = pd.DataFrame(rows, columns=['Output', 'N', 'Mean', 'MCSE'])
conv['Relative MCSE %'] = conv['MCSE'] / conv['Mean'].abs() * 100

print("MONTE CARLO CONVERGENCE CHECK (mean of 30 runs +/- MCSE)")
print("=" * 75)
for name in metrics:
    d = conv[conv['Output'] == name].set_index('N')
    cells = [f"{d.loc[n,'Mean']:.2f} +/-{d.loc[n,'MCSE']:.2f}" for n in [1000, 10000, 50000]]
    print(f"{name:<34} N=1,000: {cells[0]:<18} N=10,000: {cells[1]:<18} N=50,000: {cells[2]}")

print("\nFull table:")
print(conv.round(3).to_string(index=False))

# ---- Chart ----
rng = np.random.default_rng(42); x = retro_total(50000, rng) / 1e6
rng = np.random.default_rng(42); z = (gap_pct(50000, rng, 0.30, 5) > 0.8) * 100
n = np.arange(1, 50001)

fig, ax = plt.subplots(1, 3, figsize=(16, 4.8))
ax[0].plot(n, np.cumsum(x) / n, color='darkred')
ax[0].axhline(483.1, ls='--', color='k', lw=1, label='Thesis value: 483.1M')
ax[0].axvline(10000, color='gray', ls=':'); ax[0].set_xscale('log'); ax[0].set_ylim(470, 500)
ax[0].set_title('(a) Running mean: total retrospective gap')
ax[0].set_xlabel('Number of simulations (log scale)'); ax[0].set_ylabel('Naira Million'); ax[0].legend(fontsize=8)

ax[1].plot(n, np.cumsum(z) / n, color='navy')
ax[1].axhline(74.0, ls='--', color='k', lw=1, label='Thesis value: 74.0%')
ax[1].axvline(10000, color='gray', ls=':'); ax[1].set_xscale('log'); ax[1].set_ylim(65, 85)
ax[1].set_title('(b) Running estimate: S3 P(gap > 80%) in 2030')
ax[1].set_xlabel('Number of simulations (log scale)'); ax[1].set_ylabel('Probability (%)'); ax[1].legend(fontsize=8)

for name, col in zip(['Retro total gap 2021-2025 (N M)', 'S1 2030 mean gap (N M)',
                      'S3 2030 P(gap>80%) %', 'S4 2028 P(gap>80%) %'],
                     ['darkred', 'darkorange', 'navy', 'green']):
    d = conv[conv['Output'] == name]
    ax[2].loglog(d['N'], d['Relative MCSE %'], 'o-', color=col, label=name)
NN = np.array([500, 50000])
ax[2].loglog(NN, 12.6 * np.sqrt(500 / NN), 'k--', lw=1, label='1/sqrt(N) reference')
ax[2].axhline(1, color='gray', ls=':', lw=1); ax[2].axvline(10000, color='gray', ls=':')
ax[2].set_title('(c) Relative Monte Carlo standard error')
ax[2].set_xlabel('Number of simulations'); ax[2].set_ylabel('Relative MCSE (%)'); ax[2].legend(fontsize=7)

plt.tight_layout()
plt.show()


## F2: VaR and CVaR of the revenue gap (Table 4.21, Figure 4.10)

In [ ]:
# -- CELL: VaR and CVaR of the Revenue Gap (2026-2030) --
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df_flights = pd.read_excel("DATA.xlsx", sheet_name="Flight Records")
df_monthly = pd.read_excel("DATA.xlsx", sheet_name="Monthly Summary")
df_flights = df_flights[pd.to_numeric(df_flights['TOTAL'], errors='coerce') > 0]

inflation_data = {
    2008:0.1158, 2009:0.1254, 2010:0.1374, 2011:0.1083,
    2012:0.1222, 2013:0.0850, 2014:0.0805, 2015:0.0901,
    2016:0.1570, 2017:0.1650, 2018:0.1210, 2019:0.1140,
    2020:0.1325, 2021:0.1695, 2022:0.1885, 2023:0.2466,
    2024:0.3324, 2025:0.2333
}
rates_all = np.array(list(inflation_data.values()))
mu = rates_all.mean()
sd = rates_all.std(ddof=0)              # population SD (6.29%), as in Cell 2

A_base = np.prod([1 + inflation_data[y] for y in range(2008, 2025)])
base_movements = df_monthly[df_monthly['YEAR'] == 2025]['FLIGHTS'].sum() * 12/9
base_revenue = df_flights[df_flights['YEAR'] == 2025]['TOTAL'].sum() * 12/9
avg_charge = base_revenue / base_movements
a21 = df_monthly[df_monthly['YEAR'] == 2021]['ACTUAL_FLIGHTS'].sum()
a24 = df_monthly[df_monthly['YEAR'] == 2024]['ACTUAL_FLIGHTS'].sum()
cagr = (a24 / a21) ** (1/3) - 1

# (tariff growth g, traffic growth)
scenarios = {
    'S1 Baseline (g=0%)':        (0.00, cagr),
    'S2 Conservative (g=15%)':   (0.15, cagr),
    'S3 Moderate (g=30%)':       (0.30, cagr),
    'S4 Aggressive (g=50%)':     (0.50, cagr),
    'S5 Traffic Growth (γ=15%)': (0.00, 0.15),
}

def var_cvar(a, q):
    v = np.percentile(a, q)          # VaR = q-th percentile of the gap
    return v, a[a >= v].mean()       # CVaR = average of outcomes at or beyond VaR

N = 10000
rng = np.random.default_rng(42)
rows = []

for name, (g, gm) in scenarios.items():
    for year in range(2026, 2031):
        t = year - 2025
        fi = np.clip(rng.normal(mu, sd, (N, t)), 0.01, 0.80)
        movements = base_movements * (1 + gm) ** t
        nominal = movements * avg_charge * (1 + g) ** t
        required = movements * avg_charge * A_base * np.prod(1 + fi, axis=1)
        G = required - nominal
        gp = G / required * 100

        v95, c95 = var_cvar(G, 95)
        v99, c99 = var_cvar(G, 99)
        pv95, pc95 = var_cvar(gp, 95)
        rows.append([name, year, G.mean()/1e6, v95/1e6, c95/1e6, v99/1e6, c99/1e6,
                     gp.mean(), pv95, pc95])

risk = pd.DataFrame(rows, columns=['Scenario', 'Year', 'Mean gap (N M)',
                                   'VaR95 (N M)', 'CVaR95 (N M)',
                                   'VaR99 (N M)', 'CVaR99 (N M)',
                                   'Mean gap %', 'VaR95 gap %', 'CVaR95 gap %'])

pd.set_option('display.width', 200)
print("VaR AND CVaR OF THE REVENUE GAP (2026-2030)")
print("=" * 110)
print(risk.round(2).to_string(index=False))

print("\nSUMMARY FOR 2030")
print("-" * 80)
print(risk[risk['Year'] == 2030][['Scenario', 'Mean gap (N M)', 'VaR95 (N M)',
                                  'CVaR95 (N M)', 'CVaR95 gap %']].round(1).to_string(index=False))

# ---- Chart ----
fig, ax = plt.subplots(1, 2, figsize=(15, 5.5))
colors = ['red', 'goldenrod', 'blue', 'green', 'purple']
years = list(range(2026, 2031))

for (name, _), col in zip(scenarios.items(), colors):
    d = risk[risk['Scenario'] == name]
    ax[0].plot(years, d['Mean gap (N M)'], 'o-', color=col, label=name + ' mean')
    ax[0].plot(years, d['CVaR95 (N M)'], 's--', color=col, alpha=0.7)
ax[0].set_title('(a) Mean gap (solid) and CVaR 95% (dashed), 2026 to 2030')
ax[0].set_xlabel('Year'); ax[0].set_ylabel('Revenue gap (Naira Million)')
ax[0].set_xticks(years); ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)

d30 = risk[risk['Year'] == 2030]
x = np.arange(len(d30)); w = 0.27
ax[1].bar(x - w, d30['Mean gap (N M)'], w, label='Mean gap', color='steelblue')
ax[1].bar(x,     d30['VaR95 (N M)'],    w, label='VaR 95%',  color='orange')
ax[1].bar(x + w, d30['CVaR95 (N M)'],   w, label='CVaR 95%', color='darkred')
ax[1].set_xticks(x)
ax[1].set_xticklabels([n.split(' (')[0] for n in scenarios], fontsize=9)
ax[1].set_title('(b) Mean gap, VaR 95% and CVaR 95% in 2030')
ax[1].set_ylabel('Revenue gap (Naira Million)')
ax[1].legend(fontsize=9); ax[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()


## F3: Out-of-sample backtest (Table 4.19)

In [ ]:
# -- CELL: Out-of-Sample Backtest (fit 2021-2023, test 2024-2025) --
import numpy as np
import pandas as pd

df_flights = pd.read_excel("DATA.xlsx", sheet_name="Flight Records")
df_monthly = pd.read_excel("DATA.xlsx", sheet_name="Monthly Summary")
df_flights = df_flights[pd.to_numeric(df_flights['TOTAL'], errors='coerce') > 0]

inflation_data = {
    2008:0.1158, 2009:0.1254, 2010:0.1374, 2011:0.1083,
    2012:0.1222, 2013:0.0850, 2014:0.0805, 2015:0.0901,
    2016:0.1570, 2017:0.1650, 2018:0.1210, 2019:0.1140,
    2020:0.1325, 2021:0.1695, 2022:0.1885, 2023:0.2466,
    2024:0.3324, 2025:0.2333
}

def cumulative_factor_lagged(year):
    return np.prod([1 + inflation_data[y] for y in range(2008, year)])

# ---- 1. Fit using ONLY information available at the end of 2023 ----
train_infl = np.array([inflation_data[y] for y in range(2008, 2024)])
mu = train_infl.mean()
sd = train_infl.std(ddof=0)

train_months = df_monthly[df_monthly['YEAR'] <= 2023]['REVENUE'].values
m_mean, m_sd = train_months.mean(), train_months.std(ddof=0)

annual = df_monthly.groupby('YEAR')['REVENUE'].sum()
actual = {2024: annual[2024], 2025: annual[2025] * 12/9}   # 2025 annualised
anchor_2021 = annual[2021]

a21 = df_monthly[df_monthly['YEAR'] == 2021]['ACTUAL_FLIGHTS'].sum()
a23 = df_monthly[df_monthly['YEAR'] == 2023]['ACTUAL_FLIGHTS'].sum()
cagr = (a23 / a21) ** (1/2) - 1
base_2023 = annual[2023]

print(f"Training inflation 2008-2023: mean {mu*100:.2f}%, SD {sd*100:.2f}%")
print(f"Traffic CAGR 2021-2023 (actual flights): {cagr*100:.2f}%")
print(f"2023 base revenue: N{base_2023/1e6:.2f}M\n")

N = 10000
rng = np.random.default_rng(42)
res = {}

# Model A (comparison): stationary monthly revenue, normal fitted on 2021-2023
for y in [2024, 2025]:
    res[('A', y)] = rng.normal(m_mean, m_sd, (N, 12)).sum(axis=1)

# Model B (thesis model): frozen charges + traffic trend from 2023 base
for y in [2024, 2025]:
    h = y - 2023
    res[('B', y)] = base_2023 * (1 + cagr) ** h + rng.normal(0, m_sd * np.sqrt(12), N)

print("REVENUE FORECAST vs ACTUAL")
print("=" * 85)
for (model, y), R in res.items():
    lo, hi = np.percentile(R, [5, 95])
    err = (R.mean() - actual[y]) / actual[y] * 100
    inside = "Yes" if lo <= actual[y] <= hi else "No"
    print(f"Model {model} {y}: forecast N{R.mean()/1e6:.2f}M  90% [N{lo/1e6:.2f}M, N{hi/1e6:.2f}M]  "
          f"actual N{actual[y]/1e6:.2f}M  error {err:+.1f}%  inside: {inside}")

# ---- 2. Inflation forecast vs actual ----
print("\nINFLATION FORECAST vs ACTUAL")
print("=" * 85)
for y in [2024, 2025]:
    d = np.clip(rng.normal(mu, sd, N), 0.01, 0.80)
    lo, hi = np.percentile(d, [5, 95])
    rank = (d < inflation_data[y]).mean() * 100
    print(f"{y}: 90% [{lo*100:.1f}%, {hi*100:.1f}%]  actual {inflation_data[y]*100:.1f}%  "
          f"percentile of actual: {rank:.1f}")

# ---- 3. Revenue gap % forecast vs actual (2021-anchor framework, Model B) ----
print("\nREVENUE GAP % FORECAST vs ACTUAL")
print("=" * 85)
for y in [2024, 2025]:
    R = res[('B', y)]
    if y == 2024:
        F = cumulative_factor_lagged(2024) * np.ones(N)          # known by end of 2023
    else:
        F = cumulative_factor_lagged(2024) * (1 + np.clip(rng.normal(mu, sd, N), 0.01, 0.80))
    gap = (1 - R / (anchor_2021 * F)) * 100
    actual_gap = (1 - actual[y] / (anchor_2021 * cumulative_factor_lagged(y))) * 100
    lo, hi = np.percentile(gap, [5, 95])
    inside = "Yes" if lo <= actual_gap <= hi else "No"
    print(f"{y}: forecast {gap.mean():.2f}%  90% [{lo:.2f}%, {hi:.2f}%]  "
          f"actual {actual_gap:.2f}%  inside: {inside}")


## F4: AR(1) inflation model: fit, diagnostics and backtest (Section 4.8.6)

In [ ]:
# -- CELL A: AR(1) Inflation Model - Fit, Diagnostics and Backtest --
import numpy as np
from scipy import stats

inflation_data = {
    2008:0.1158, 2009:0.1254, 2010:0.1374, 2011:0.1083,
    2012:0.1222, 2013:0.0850, 2014:0.0805, 2015:0.0901,
    2016:0.1570, 2017:0.1650, 2018:0.1210, 2019:0.1140,
    2020:0.1325, 2021:0.1695, 2022:0.1885, 2023:0.2466,
    2024:0.3324, 2025:0.2333
}
pi = np.array(list(inflation_data.values()))

def fit_ar1(x):
    """OLS fit of pi_t = c + phi * pi_(t-1) + e_t"""
    y, X = x[1:], np.column_stack([np.ones(len(x) - 1), x[:-1]])
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    resid = y - X @ beta
    n, k = len(y), 2
    sigma = np.sqrt(resid @ resid / (n - k))
    se = sigma * np.sqrt(np.diag(np.linalg.inv(X.T @ X)))
    t = beta / se
    p = 2 * stats.t.sf(np.abs(t), n - k)
    r2 = 1 - (resid @ resid) / ((y - y.mean()) @ (y - y.mean()))
    llf = -n / 2 * (np.log(2 * np.pi * (resid @ resid) / n) + 1)
    return beta, se, t, p, sigma, r2, resid, llf

def ljung_box(x, lags=(1, 2, 3)):
    x = x - x.mean(); n = len(x); d = x @ x
    out = []
    for h in lags:
        q = n * (n + 2) * sum((x[k:] @ x[:-k] / d) ** 2 / (n - k) for k in range(1, h + 1))
        out.append((h, q, stats.chi2.sf(q, h)))
    return out

# ---- 1. Fit on 2008-2025 ----
beta, se, t, p, sigma, r2, resid, llf = fit_ar1(pi)
c, phi = beta
print("AR(1) INFLATION MODEL (OLS, 2008-2025)")
print("=" * 60)
print(f"{'':<10}{'Coef':>10}{'Std err':>10}{'t':>8}{'p-value':>10}")
print(f"{'c':<10}{c:>10.4f}{se[0]:>10.4f}{t[0]:>8.3f}{p[0]:>10.4f}")
print(f"{'phi':<10}{phi:>10.4f}{se[1]:>10.4f}{t[1]:>8.3f}{p[1]:>10.4f}")
print(f"Residual SD (sigma): {sigma*100:.2f}%   R-squared: {r2:.3f}")
print(f"Long-run mean c/(1-phi): {c/(1-phi)*100:.2f}%")
print(f"Lag-1 autocorrelation of inflation: {np.corrcoef(pi[:-1], pi[1:])[0,1]:.3f}")

# ---- 2. Model comparison: AR(1) vs independent normal (same 17 years) ----
y = pi[1:]; n = len(y)
llf_iid = stats.norm.logpdf(y, y.mean(), y.std(ddof=0)).sum()
print("\nMODEL COMPARISON")
print("-" * 60)
print(f"AIC  AR(1): {2*3 - 2*llf:.2f}    Normal: {2*2 - 2*llf_iid:.2f}")
print(f"BIC  AR(1): {3*np.log(n) - 2*llf:.2f}    Normal: {2*np.log(n) - 2*llf_iid:.2f}")

# ---- 3. Diagnostics ----
print("\nLJUNG-BOX TEST (H0: no autocorrelation)")
print("-" * 60)
for h, q, pv in ljung_box(pi):
    print(f"Raw inflation, lag {h}: Q = {q:.3f}, p = {pv:.4f}")
for h, q, pv in ljung_box(resid):
    print(f"AR(1) residuals, lag {h}: Q = {q:.3f}, p = {pv:.4f}")
sw = stats.shapiro(resid)
print(f"Shapiro-Wilk on residuals: W = {sw.statistic:.4f}, p = {sw.pvalue:.4f}")

# ---- 4. Backtest: fit on 2008-2023, forecast 2024 and 2025 ----
bt, _, _, _, s_bt, _, _, _ = fit_ar1(pi[:-2])
rng = np.random.default_rng(42)
f24 = bt[0] + bt[1] * inflation_data[2023] + rng.normal(0, s_bt, 10000)
f25 = bt[0] + bt[1] * f24 + rng.normal(0, s_bt, 10000)
print("\nAR(1) BACKTEST (fit 2008-2023)")
print("-" * 60)
for yr, f in [(2024, f24), (2025, f25)]:
    lo, hi = np.percentile(f, [5, 95])
    print(f"{yr}: 90% [{lo*100:.1f}%, {hi*100:.1f}%]  actual {inflation_data[yr]*100:.1f}%  "
          f"percentile of actual: {(f < inflation_data[yr]).mean()*100:.1f}")


## F5: Projections under AR(1) and normal inflation (Table 4.22, Figure 4.11)

In [ ]:
# -- CELL B: Projections 2026-2030 under AR(1) vs Normal inflation --
# Run CELL A first (it provides inflation_data, c, phi and sigma)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pi = np.array(list(inflation_data.values()))
mu, sd = pi.mean(), pi.std(ddof=0)                     # thesis normal model
A_base = np.prod([1 + inflation_data[y] for y in range(2008, 2025)])

N = 10000
rng = np.random.default_rng(42)

# Normal (thesis) paths
iid = np.clip(rng.normal(mu, sd, (N, 5)), 0.01, 0.80)

# AR(1) paths, starting from the latest observed rate (2025 = 23.33%)
ar = np.zeros((N, 5))
prev = np.full(N, inflation_data[2025])
for t in range(5):
    prev = np.clip(c + phi * prev + rng.normal(0, sigma, N), 0.01, 0.80)
    ar[:, t] = prev

g_values = [0, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]
rows = []
for g in g_values:
    for name, P in [('Normal (thesis)', iid), ('AR(1)', ar)]:
        F = A_base * np.cumprod(1 + P, axis=1)
        for t in range(5):
            gap = 1 - (1 + g) ** (t + 1) / F[:, t]
            v95 = np.percentile(gap, 95)
            rows.append([g, name, 2026 + t, gap.mean()*100, (gap > 0.8).mean()*100,
                         v95*100, gap[gap >= v95].mean()*100])

res = pd.DataFrame(rows, columns=['g', 'Inflation model', 'Year', 'Mean gap %',
                                  'P(gap>80%) %', 'VaR95 gap %', 'CVaR95 gap %'])

print("MEAN INFLATION PATH 2026-2030")
print(f"Normal: {np.round(iid.mean(0)*100, 2)}")
print(f"AR(1):  {np.round(ar.mean(0)*100, 2)}\n")

pd.set_option('display.width', 200)
table = res[res['Year'].isin([2028, 2030])].pivot_table(
    index=['g', 'Year'], columns='Inflation model',
    values=['Mean gap %', 'P(gap>80%) %']).round(1)
print("GAP % AND P(UNSUSTAINABLE): NORMAL vs AR(1)")
print(table)

print("\nFIRST YEAR MEAN GAP FALLS BELOW 80%")
print("-" * 50)
for g in g_values:
    out = []
    for name in ['Normal (thesis)', 'AR(1)']:
        d = res[(res['g'] == g) & (res['Inflation model'] == name) & (res['Mean gap %'] < 80)]
        out.append(str(d['Year'].min()) if len(d) else 'Never')
    print(f"g = {g*100:>3.0f}%:  Normal {out[0]:<6}  AR(1) {out[1]}")

# ---- Chart ----
fig, ax = plt.subplots(1, 2, figsize=(15, 5.3))
hist = list(range(2018, 2026)); fy = list(range(2026, 2031))
ax[0].plot(hist, [inflation_data[y]*100 for y in hist], 'ko-', label='Actual (NBS)')
ax[0].plot(fy, iid.mean(0)*100, 's--', color='gray', label='Normal model mean')
ax[0].plot(fy, ar.mean(0)*100, 'o-', color='darkred', label='AR(1) mean')
ax[0].fill_between(fy, np.percentile(ar, 5, 0)*100, np.percentile(ar, 95, 0)*100,
                   color='darkred', alpha=0.15, label='AR(1) 90% band')
ax[0].fill_between(fy, np.percentile(iid, 5, 0)*100, np.percentile(iid, 95, 0)*100,
                   color='gray', alpha=0.15, label='Normal 90% band')
ax[0].set_title('(a) Inflation paths: normal vs AR(1)')
ax[0].set_xlabel('Year'); ax[0].set_ylabel('Annual inflation (%)')
ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)

for g, col in [(0, 'red'), (0.30, 'blue'), (0.35, 'teal'), (0.50, 'green')]:
    for name, ls in [('Normal (thesis)', '--'), ('AR(1)', '-')]:
        d = res[(res['g'] == g) & (res['Inflation model'] == name)]
        ax[1].plot(d['Year'], d['Mean gap %'], ls, marker='o', color=col,
                   label=f'g={int(g*100)}%, {name}')
ax[1].axhline(80, color='k', ls=':', label='80% threshold')
ax[1].set_xticks(fy)
ax[1].set_title('(b) Mean gap %: normal (dashed) vs AR(1) (solid)')
ax[1].set_xlabel('Year'); ax[1].set_ylabel('Gap %')
ax[1].legend(fontsize=7, ncol=2); ax[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()


## F6: Fat-tailed (Student-t) inflation check (Section 4.8.6)

In [ ]:
# -- CELL C: Fat-tailed (Student-t) inflation robustness check --
# Run CELL A first (it provides inflation_data, c, phi, sigma and resid)
import numpy as np
import pandas as pd
from scipy import stats, optimize

pi = np.array(list(inflation_data.values()))
mu, sd = pi.mean(), pi.std(ddof=0)
A_base = np.prod([1 + inflation_data[y] for y in range(2008, 2025)])

# ---- 1. Do the data show fat tails? ----
print("TAIL CHECK ON INFLATION (2008-2025) AND AR(1) RESIDUALS")
print("=" * 60)
for lab, x in [('Inflation levels', pi), ('AR(1) residuals', resid)]:
    jb = stats.jarque_bera(x)
    print(f"{lab:<18} skew {stats.skew(x):6.3f}  excess kurtosis {stats.kurtosis(x):6.3f}  "
          f"Jarque-Bera p = {jb.pvalue:.3f}")

# MLE of Student-t on AR(1) residuals (location 0)
def nll(params):
    df, s = np.exp(params)
    return -stats.t.logpdf(resid, df, 0, s).sum()
opt = optimize.minimize(nll, [np.log(10), np.log(sigma)], method='Nelder-Mead')
df_hat, s_hat = np.exp(opt.x)
ll_t = -opt.fun
ll_n = stats.norm.logpdf(resid, 0, resid.std(ddof=0)).sum()
print(f"\nStudent-t fit to residuals: df = {df_hat:.1f}, scale = {s_hat*100:.2f}%")
print(f"AIC  Normal: {2*1 - 2*ll_n:.2f}   Student-t: {2*2 - 2*ll_t:.2f}")

# ---- 2. Stress test: AR(1) with t shocks, same variance as normal ----
def ar_paths(df, N=10000, seed=42):
    rng = np.random.default_rng(seed)
    if df is None:
        e = rng.normal(0, sigma, (N, 5))
    else:
        e = rng.standard_t(df, (N, 5)) * sigma * np.sqrt((df - 2) / df)
    P = np.zeros((N, 5)); prev = np.full(N, inflation_data[2025])
    for t in range(5):
        prev = np.clip(c + phi * prev + e[:, t], 0.01, 0.80)
        P[:, t] = prev
    return P

models = {'AR(1) normal': None, 'AR(1) t, df=5': 5, 'AR(1) t, df=3': 3}
paths = {k: ar_paths(v) for k, v in models.items()}

rows = []
for g in [0, 0.15, 0.30, 0.35, 0.40, 0.45, 0.50]:
    for name, P in paths.items():
        F = A_base * np.cumprod(1 + P, axis=1)
        for t in range(5):
            gap = 1 - (1 + g) ** (t + 1) / F[:, t]
            v95 = np.percentile(gap, 95)
            rows.append([g, name, 2026 + t, gap.mean()*100, (gap > 0.8).mean()*100,
                         v95*100, gap[gap >= v95].mean()*100, np.percentile(P[:, t], 99)*100])
res_t = pd.DataFrame(rows, columns=['g', 'Model', 'Year', 'Mean gap %', 'P(gap>80%) %',
                                    'VaR95 gap %', 'CVaR95 gap %', 'Infl 99th pct %'])

pd.set_option('display.width', 250); pd.set_option('display.max_columns', 20)
print("\n2030 RESULTS: NORMAL vs FAT-TAILED SHOCKS")
print(res_t[res_t['Year'] == 2030].pivot_table(
    index='g', columns='Model',
    values=['Mean gap %', 'P(gap>80%) %', 'CVaR95 gap %']).round(1))

print("\nFIRST YEAR MEAN GAP FALLS BELOW 80%")
print("-" * 60)
for g in sorted(res_t['g'].unique()):
    out = []
    for name in models:
        d = res_t[(res_t['g'] == g) & (res_t['Model'] == name) & (res_t['Mean gap %'] < 80)]
        out.append(str(d['Year'].min()) if len(d) else 'Never')
    print(f"g = {g*100:>3.0f}%:  " + "  ".join(f"{m}: {o:<6}" for m, o in zip(models, out)))


## F7: Imputation robustness check (Table 4.20)

In [ ]:
# -- CELL D: Imputation Robustness Check (13 imputed months, 2021-2022) --
# Needs the MASTER SETUP CELL (df_monthly, inflation_data, cumulative_factor_lagged)
import numpy as np
import pandas as pd

m = df_monthly.copy()
m['IMPUTED'] = m['IMPUTED_FLIGHTS'] > 0
print(f"Imputed months: {m['IMPUTED'].sum()} "
      f"({m[m.IMPUTED & (m.YEAR==2021)].shape[0]} in 2021, {m[m.IMPUTED & (m.YEAR==2022)].shape[0]} in 2022)")

# Thesis values (from Cells 5, 8 and 11)
THESIS_RETRO_MC, THESIS_CI = 483.1, (337.5, 657.0)    # N million
THESIS_S1_2030 = 306.72                                # N million
years = [2021, 2022, 2023, 2024, 2025]

def outputs(rev, fl2021):
    """Every result that depends on 2021-2022 monthly data"""
    annual = {y: rev[m.YEAR == y].sum() for y in years}
    annual[2025] *= 12 / 9
    anchor = annual[2021]
    det_total = sum(anchor * cumulative_factor_lagged(y) - annual[y] for y in years) / 1e6
    k = anchor / m.loc[m.YEAR == 2021, 'REVENUE'].sum()   # MC retro gap is linear in the anchor
    cagr = (m.loc[m.YEAR == 2024, 'ACTUAL_FLIGHTS'].sum() / fl2021) ** (1/3) - 1
    s1 = THESIS_S1_2030 * ((1 + cagr) / (1 + cagr_thesis)) ** 5
    return anchor / 1e6, det_total, THESIS_RETRO_MC * k, cagr * 100, s1

cagr_thesis = (m.loc[m.YEAR == 2024, 'ACTUAL_FLIGHTS'].sum() /
               m.loc[m.YEAR == 2021, 'ACTUAL_FLIGHTS'].sum()) ** (1/3) - 1
rev0 = m['REVENUE'].values.astype(float)
act21 = m.loc[m.YEAR == 2021, 'ACTUAL_FLIGHTS'].sum()
tot21 = m.loc[m.YEAR == 2021, 'FLIGHTS'].sum()

rows = []
# A. Thesis as submitted
rows.append(['A. Thesis (as submitted)', *outputs(rev0, act21)])

# B. Drop imputed months, annualise from actual months only
revB = rev0.copy()
for y in [2021, 2022]:
    idx = (m.YEAR == y).values
    act = idx & ~m.IMPUTED.values
    revB[idx & m.IMPUTED.values] = rev0[act].mean()
rows.append(['B. Actual months only, annualised', *outputs(revB, act21)])

# C/D. Imputed months 20% lower / 20% higher
for lab, f in [('C. Imputed months -20%', 0.8), ('D. Imputed months +20%', 1.2)]:
    r = rev0.copy(); r[m.IMPUTED.values] *= f
    rows.append([lab, *outputs(r, act21)])

# E. CAGR with 2021 actual flights annualised (715 flights cover only 9 months)
rows.append(['E. CAGR, 2021 flights annualised', *outputs(rev0, act21 * 12 / 9)])
# F. CAGR on full-year 2021 flights including imputed months (937)
rows.append(['F. CAGR, full-year 2021 flights', *outputs(rev0, tot21)])

res_imp = pd.DataFrame(rows, columns=['Treatment', 'Anchor 2021 (NM)', 'Retro gap det. (NM)',
                                      'Retro gap MC (NM)', 'Traffic CAGR %', 'S1 2030 gap (NM)'])

# G. Bootstrap: refill each imputed month with a random actual month (2021-2023)
pool = rev0[(~m.IMPUTED.values) & (m.YEAR <= 2023).values]
rng = np.random.default_rng(42)
B = 10000
boot = np.array([outputs(np.where(m.IMPUTED.values,
                                  rng.choice(pool, len(rev0)), rev0), act21)
                 for _ in range(B)])

pd.set_option('display.width', 250); pd.set_option('display.max_columns', 20)
print("\nIMPUTATION ROBUSTNESS: RESULTS UNDER EACH TREATMENT")
print("=" * 110)
print(res_imp.round(2).to_string(index=False))

print(f"\nBOOTSTRAP ({B:,} refills from {len(pool)} actual months, 2021-2023)")
print("-" * 70)
for j, name in enumerate(res_imp.columns[1:4]):
    lo, hi = np.percentile(boot[:, j], [5, 95])
    print(f"{name:<22} mean {boot[:, j].mean():8.2f}   90% range {lo:8.2f} to {hi:8.2f}")

print("\nUNAFFECTED BY IMPUTATION (by construction)")
print("-" * 70)
print("All projected gap % values, P(gap > 80%), the 35%/40% minimum tariff growth,")
print("VaR/CVaR gap %: these depend only on inflation and g, not on 2021-2022 revenue.")
print(f"\nThesis retro MC 90% CI: N{THESIS_CI[0]}M to N{THESIS_CI[1]}M")


## F8: Traffic growth rate check (Section 4.8.4)

In [ ]:
# -- CELL E: Corrected traffic growth rate (like-for-like months) --
# Needs the MASTER SETUP CELL
import numpy as np
import pandas as pd
from scipy import stats

m = df_monthly.copy()
m['t'] = np.arange(len(m))                    # month index, Jan 2021 = 0
act = m[m['IMPUTED_FLIGHTS'] == 0]            # 44 months with actual records only
flights_per_month = lambda y: act.loc[act.YEAR == y, 'FLIGHTS'].mean()

# 1. Thesis rate: 715 flights (9 months of 2021) vs 725 flights (12 months of 2024)
cagr_thesis = (act.loc[act.YEAR == 2024, 'FLIGHTS'].sum() /
               act.loc[act.YEAR == 2021, 'FLIGHTS'].sum()) ** (1/3) - 1
# 2. Like-for-like: average flights per actual month, 2021 vs 2024
cagr_2124 = (flights_per_month(2024) / flights_per_month(2021)) ** (1/3) - 1
# 3. Like-for-like: 2021 vs 2025
cagr_2125 = (flights_per_month(2025) / flights_per_month(2021)) ** (1/4) - 1
# 4. Log-linear trend on all 44 actual months (uses every data point)
reg = stats.linregress(act['t'], np.log(act['FLIGHTS']))
cagr_trend = np.exp(12 * reg.slope) - 1
se_annual = 12 * reg.stderr
ci = (np.exp(12 * reg.slope - 1.96 * se_annual) - 1, np.exp(12 * reg.slope + 1.96 * se_annual) - 1)
# 5. External check: Yola passengers (NBS 2021: 175,699; FAAN 2024: 161,847)
cagr_pax = (161847 / 175699) ** (1/3) - 1

print("TRAFFIC GROWTH RATE: THESIS vs CORRECTED")
print("=" * 70)
print(f"Avg flights per actual month: 2021 {flights_per_month(2021):.1f}, "
      f"2024 {flights_per_month(2024):.1f}, 2025 {flights_per_month(2025):.1f}")
rates = {
    'Thesis (9 vs 12 months)': cagr_thesis,
    'Like-for-like 2021-2024': cagr_2124,
    'Like-for-like 2021-2025': cagr_2125,
    'Log-linear trend, 44 months': cagr_trend,
    'Yola passengers 2021-2024': cagr_pax,
}
for k, v in rates.items():
    print(f"{k:<30} {v*100:+6.2f}% per year")
print(f"Trend 95% CI: {ci[0]*100:+.2f}% to {ci[1]*100:+.2f}%   slope p-value = {reg.pvalue:.4f}")

# Effect on 2030 naira gaps (gap % does not depend on traffic)
mu, sd = np.mean(list(inflation_data.values())), np.std(list(inflation_data.values()))
rng = np.random.default_rng(42)
F = A_base * np.prod(1 + np.clip(rng.normal(mu, sd, (10000, 5)), 0.01, 0.80), axis=1)
print("\n2030 MEAN REVENUE GAP (N MILLION) UNDER EACH TRAFFIC RATE")
print("-" * 70)
print(f"{'Traffic rate':<30}{'S1 g=0%':>10}{'S3 g=30%':>10}{'S4 g=50%':>10}{'Gap % S1':>10}")
for k, c in rates.items():
    mov = base_movements * (1 + c) ** 5
    out = [np.mean(mov * avg_charge_per_flight * (F - (1 + g) ** 5)) / 1e6 for g in (0, .3, .5)]
    gp = np.mean(1 - 1 / F) * 100
    print(f"{k:<30}{out[0]:>10.1f}{out[1]:>10.1f}{out[2]:>10.1f}{gp:>10.1f}")


## F9: Stronger goodness-of-fit tests (Table 4.18)

In [ ]:
# -- CELL F: Stronger goodness-of-fit tests (Lilliefors, Anderson-Darling, Shapiro-Wilk) --
# Needs the MASTER SETUP CELL. scipy >= 1.11 for stats.goodness_of_fit
import numpy as np
import pandas as pd
from scipy import stats

data = {
    'Monthly revenue (57 months)': df_monthly['REVENUE'].values.astype(float),
    'Monthly flights (57 months)': df_monthly['FLIGHTS'].values.astype(float),
    'Monthly revenue, actual months only (44)':
        df_monthly.loc[df_monthly['IMPUTED_FLIGHTS'] == 0, 'REVENUE'].values.astype(float),
    'Annual inflation 2008-2025 (18)': np.array(list(inflation_data.values())),
}

rows = []
for name, x in data.items():
    # Plain K-S with estimated parameters (as in the thesis) - p-value is too generous
    ks = stats.kstest(x, stats.norm(x.mean(), x.std()).cdf)
    # Lilliefors: K-S with the p-value corrected for estimated parameters (Monte Carlo)
    lil = stats.goodness_of_fit(stats.norm, x, statistic='ks', n_mc_samples=9999,
                                random_state=np.random.default_rng(42))
    ad = stats.goodness_of_fit(stats.norm, x, statistic='ad', n_mc_samples=9999,
                               random_state=np.random.default_rng(42))
    sw = stats.shapiro(x)
    # Normal vs lognormal by AIC
    aic_n = 2*2 - 2*stats.norm.logpdf(x, *stats.norm.fit(x)).sum()
    aic_l = 2*2 - 2*stats.lognorm.logpdf(x, *stats.lognorm.fit(x, floc=0)).sum()
    rows.append([name, ks.pvalue, lil.pvalue, ad.pvalue, sw.pvalue, aic_n - aic_l])

res_gof = pd.DataFrame(rows, columns=['Variable', 'K-S p (thesis)', 'Lilliefors p',
                                      'Anderson-Darling p', 'Shapiro-Wilk p',
                                      'AIC(normal) - AIC(lognormal)'])
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 20)
print("GOODNESS-OF-FIT TO THE NORMAL DISTRIBUTION")
print("=" * 120)
print(res_gof.round(4).to_string(index=False))
print("\nRule: p > 0.05 means normality is not rejected.")
print("Positive AIC difference means the lognormal fits better; above 2 is a meaningful gap.")

# ---- Does a lognormal inflation model change the thesis results? ----
infl = data['Annual inflation 2008-2025 (18)']
shape, loc, scale = stats.lognorm.fit(infl, floc=0)
rng = np.random.default_rng(42)
N = 10000
paths = {
    'Normal (thesis)': np.clip(rng.normal(infl.mean(), infl.std(), (N, 5)), 0.01, 0.80),
    'Lognormal': np.clip(stats.lognorm.rvs(shape, 0, scale, size=(N, 5), random_state=rng), 0.01, 0.80),
}
print("\nPROJECTED MEAN GAP % AND P(GAP > 80%), 2030: NORMAL vs LOGNORMAL INFLATION")
print("-" * 80)
print(f"{'g':<6}{'Normal gap %':>14}{'Lognormal gap %':>17}{'Normal P%':>12}{'Lognormal P%':>15}")
for g in [0, 0.15, 0.30, 0.35, 0.40, 0.50]:
    out = []
    for P in paths.values():
        gap = 1 - (1 + g) ** 5 / (A_base * np.prod(1 + P, axis=1))
        out += [gap.mean() * 100, (gap > 0.8).mean() * 100]
    print(f"{g*100:<6.0f}{out[0]:>14.1f}{out[2]:>17.1f}{out[1]:>12.1f}{out[3]:>15.1f}")


## F10: Sobol global sensitivity analysis (Table 4.23)

In [ ]:
# -- CELL G: Global sensitivity analysis (Sobol indices, Saltelli/Jansen estimators) --
# Needs the MASTER SETUP CELL (A_base, base_movements, avg_charge_per_flight)
import numpy as np
import pandas as pd

# Inputs varied together over plausible ranges (uniform)
bounds = {
    'Tariff growth g':          (0.00, 0.50),    # S1 to S4
    'Mean inflation 2026-30':   (0.10, 0.25),    # historical low to recent high
    'Inflation volatility':     (0.02, 0.10),    # SD of yearly shocks
    'Traffic growth':           (-0.075, 0.15),  # trend CI lower bound to S5
}
names = list(bounds); k = len(names)
lo = np.array([b[0] for b in bounds.values()]); hi = np.array([b[1] for b in bounds.values()])

N = 2 ** 14
rng = np.random.default_rng(42)
A = lo + (hi - lo) * rng.random((N, k))
B = lo + (hi - lo) * rng.random((N, k))
Z = rng.standard_normal((N, 5))              # common inflation shocks (kept fixed across matrices)

def model(X):
    g, mu, sd, c = X.T
    infl = np.clip(mu[:, None] + sd[:, None] * Z, 0.01, 0.80)
    F = A_base * np.prod(1 + infl, axis=1)
    mov = base_movements * (1 + c) ** 5
    gap_pct = 1 - (1 + g) ** 5 / F
    gap_naira = mov * avg_charge_per_flight * (F - (1 + g) ** 5) / 1e6
    return gap_pct * 100, gap_naira

yA, yB = model(A), model(B)
rows = []
for out, label in enumerate(['2030 gap %', '2030 gap (N million)']):
    fA, fB = yA[out], yB[out]
    var = np.var(np.concatenate([fA, fB]))
    for i, n in enumerate(names):
        ABi = A.copy(); ABi[:, i] = B[:, i]
        fABi = model(ABi)[out]
        S1 = np.mean(fB * (fABi - fA)) / var                 # Saltelli 2010 first order
        ST = 0.5 * np.mean((fA - fABi) ** 2) / var           # Jansen total effect
        rows.append([label, n, S1, ST])

sob = pd.DataFrame(rows, columns=['Output', 'Input', 'First-order S1', 'Total ST'])
pd.set_option('display.width', 200)
print(f"SOBOL GLOBAL SENSITIVITY INDICES (N = {N:,} per matrix, {N*(k+2):,} model runs per output)")
print("=" * 75)
for out in sob['Output'].unique():
    print(f"\n{out}")
    print(sob[sob.Output == out].drop(columns='Output').round(3).to_string(index=False))
print("\nS1 = share of output variance explained by that input alone.")
print("ST = share including its interactions with the other inputs.")


## F11: Comparison with northern airports (Table 4.24)

In [ ]:
# -- CELL H: Yola against northern airports (benchmark) --
# Self-contained. Sources: NBS Air Transportation Data 2021; FAAN 2024 via Daily Trust (8 Jan 2026);
# NATCA 2023 movements via Daily Trust (3 Dec 2024)
import numpy as np
import pandas as pd

bench = pd.DataFrame({
    'Airport':   ['Yola', 'Maiduguri', 'Jos', 'Sokoto', 'Kano', 'Kaduna', 'Katsina', 'Ilorin', 'Gombe', 'Bauchi'],
    'Pax_2021':  [175699, 197898, 57020, 137511, 545749, 73110, 30313, 148109, np.nan, np.nan],
    'Pax_2024':  [161847, 180149, 39496, 144047, 514460, 57156, 44830, 143562, 32778, 21897],
    'Mov_2023':  [2904, 8660, 1444, 2405, 11133, np.nan, 1300, 5094, 772, 969],
})
nat_2021, nat_2024 = 13006481, 12543153

bench['Pax change 21-24 %'] = (bench.Pax_2024 / bench.Pax_2021 - 1) * 100
bench['Pax CAGR 21-24 %'] = ((bench.Pax_2024 / bench.Pax_2021) ** (1/3) - 1) * 100
bench['Pax per movement*'] = bench.Pax_2024 / bench.Mov_2023
print("YOLA AGAINST NORTHERN AIRPORTS")
print("=" * 95)
print(bench.round(1).to_string(index=False))
print("* 2024 passengers over 2023 movements (latest years available); a rough load indicator only.")

grp = bench.dropna(subset=['Pax_2021'])
north = (grp.Pax_2024.sum() / grp.Pax_2021.sum()) ** (1/3) - 1
print(f"\nNational domestic passengers CAGR 2021-24: {((nat_2024/nat_2021)**(1/3)-1)*100:+.2f}%")
print(f"Northern group (8 airports) CAGR 2021-24:  {north*100:+.2f}%")
print(f"Yola CAGR 2021-24:                          {bench.loc[0,'Pax CAGR 21-24 %']:+.2f}%")
print(f"Yola longer run (221,601 in 2018 to 161,847 in 2024): {((161847/221601)**(1/6)-1)*100:+.2f}% per year")
print(f"Yola rank by 2024 passengers among these 10: "
      f"{int(bench.Pax_2024.rank(ascending=False)[0])}")

# Plausibility of the traffic scenarios
print("\nTRAFFIC SCENARIO PLAUSIBILITY")
print("-" * 60)
for lab, g in [('Thesis baseline', 0.0046), ('S5 traffic growth', 0.15)]:
    print(f"{lab:<20} {g*100:+.2f}%/yr -> 2030 traffic x{(1+g)**5:.2f} of 2025")
print(f"No airport in this group grew faster than "
      f"{bench['Pax CAGR 21-24 %'].max():.1f}% a year in 2021-2024.")


## F12: Exchange rate risk (Table 4.25)

In [ ]:
# -- CELL I: Foreign-exchange (FX) risk: revenue in US dollar terms --
# Needs the MASTER SETUP CELL. FX = CBN official annual average NGN per USD
import numpy as np
import pandas as pd
from scipy import stats

fx = {2008:117.00, 2009:146.82, 2010:148.31, 2011:151.83, 2012:155.44, 2013:155.25,
      2014:156.48, 2015:192.55, 2016:253.00, 2017:305.29, 2018:305.58, 2019:306.42,
      2020:356.40, 2021:398.87, 2022:423.32, 2023:644.02, 2024:1478.55, 2025:1517.87}

# 1. Historical revenue in USD (2025 annualised)
rev = df_monthly.groupby('YEAR')['REVENUE'].sum().astype(float)
rev[2025] *= 12 / 9
hist = pd.DataFrame({'Revenue (NM)': rev / 1e6,
                     'NGN per USD': [fx[y] for y in rev.index],
                     'Revenue (USD 000)': [rev[y] / fx[y] / 1e3 for y in rev.index],
                     'USD per flight': [rev[y] / fx[y] / (df_monthly.loc[df_monthly.YEAR == y, 'FLIGHTS'].sum()
                                        * (12/9 if y == 2025 else 1)) for y in rev.index]})
print("HISTORICAL REVENUE IN NAIRA AND US DOLLARS")
print("=" * 70)
print(hist.round(1).to_string())
print(f"USD value lost 2021-2025: {(1 - hist.iloc[-1, 2] / hist.iloc[0, 2]) * 100:.1f}%")

# 2. Joint model of inflation and naira depreciation (2009-2025)
yrs = list(range(2009, 2026))
infl = np.array([inflation_data[y] for y in yrs])
dep = np.array([np.log(fx[y] / fx[y - 1]) for y in yrs])     # log depreciation
r, p = stats.pearsonr(infl, dep)
print(f"\nMean log depreciation {dep.mean()*100:.1f}%/yr, SD {dep.std()*100:.1f}%; "
      f"correlation with inflation r = {r:.2f} (p = {p:.3f})")
mean_vec = [infl.mean(), dep.mean()]
cov = np.cov(np.vstack([infl, dep]), ddof=0)

# 3. Projections 2026-2030 in USD
N = 10000
rng = np.random.default_rng(42)
draws = rng.multivariate_normal(mean_vec, cov, size=(N, 5))
fx_path = fx[2025] * np.exp(np.cumsum(draws[:, :, 1], axis=1))
cagr = 0.0046
usd_2025 = rev[2025] / fx[2025]
print("\nPROJECTED REVENUE IN USD 000 (MEAN) AND P(USD REVENUE BELOW 2025 LEVEL)")
print("-" * 80)
print(f"{'Scenario':<22}" + "".join(f"{y:>10}" for y in range(2026, 2031)) + f"{'P<2025 in 2030':>16}")
for lab, g in [('S1 frozen (g=0%)', 0), ('S2 g=15%', .15), ('S3 g=30%', .30), ('S4 g=50%', .50)]:
    t = np.arange(1, 6)
    usd = base_revenue * (1 + cagr) ** t * (1 + g) ** t / fx_path / 1e3
    print(f"{lab:<22}" + "".join(f"{v:>10.1f}" for v in usd.mean(0)) +
          f"{(usd[:, -1] < usd_2025 / 1e3).mean()*100:>15.1f}%")
print(f"\n2025 USD revenue: {usd_2025/1e3:.1f} thousand;  2021: {hist.iloc[0, 2]:.1f} thousand")
usd_s4 = base_revenue * (1 + cagr) ** 5 * 1.5 ** 5 / fx_path[:, -1]
print(f"P(S4 regains 2021 USD revenue by 2030): {(usd_s4 >= hist.iloc[0, 2] * 1e3).mean()*100:.1f}%")
print(f"Median NGN per USD in 2030: {np.median(fx_path[:, -1]):,.0f}")
